In [0]:
import json
import hashlib
from pathlib import Path
from pyspark.sql import functions as F

batch_id = "batch_001"
batch_path = Path(
    f"/Volumes/workspace/migration_dev/landing/{batch_id}"
)
csv_path = batch_path / "metadata" / "statements.csv"
manifest_path = batch_path / "manifest.json"

# Only ingest a batch declared complete by the source.
if not manifest_path.is_file():
    raise RuntimeError("Batch is not ready: manifest is missing")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

if manifest["status"] != "READY" or manifest["batch_id"] != batch_id:
    raise RuntimeError("Manifest status or batch ID is invalid")

actual_csv_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()

if actual_csv_hash != manifest["metadata_sha256"]:
    raise RuntimeError("CSV has changed since the manifest was created")

# Preserve source values as strings in Bronze.
source_schema = """
    batch_id STRING,
    document_id STRING,
    account_number STRING,
    customer_name STRING,
    document_type STRING,
    statement_date STRING,
    file_name STRING,
    pdf_path STRING,
    source_sha256 STRING,
    _rescued_data STRING
"""

state_path = (
    "/Volumes/workspace/migration_dev/checkpoints/"
    f"bronze_metadata/{batch_id}"
)

raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")
    .option("cloudFiles.schemaLocation", f"{state_path}/schema")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(source_schema)
    .load(str(batch_path / "metadata"))
)

bronze = raw.select(
    "*",
    F.col("_metadata.file_path").alias("source_file"),
    F.col("_metadata.file_modification_time")
        .alias("source_modification_time"),
    F.current_timestamp().alias("ingestion_timestamp"),
)

query = (
    bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{state_path}/stream")
    .trigger(availableNow=True)
    .toTable("workspace.migration_dev.bronze_metadata")
)

query.awaitTermination()

saved = (
    spark.table("workspace.migration_dev.bronze_metadata")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Bronze records for {batch_id}: {saved.count()}")
display(saved.orderBy("document_id").limit(5))

Bronze records for batch_001: 100


batch_id,document_id,account_number,customer_name,document_type,statement_date,file_name,pdf_path,source_sha256,_rescued_data,source_file,source_modification_time,ingestion_timestamp
batch_001,DOC000001,TEST00000001,Sample Customer 001,ACCOUNT_STATEMENT,2026-09-30,DOC000001.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000001.pdf,12298c6adcbc1f0ffed046f3c14ab064da327b4fab519c9e25014e6a8182c431,null,/Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv,2026-10-13T14:41:42.000Z,2026-10-13T14:51:21.115Z
batch_001,DOC000002,TEST00000002,Sample Customer 002,ACCOUNT_STATEMENT,2026-09-30,DOC000002.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000002.pdf,e45b766bfe7a3f38c7303239ffab722e244474ccb300ef84529b48197cbb83ce,null,/Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv,2026-10-13T14:41:42.000Z,2026-10-13T14:51:21.115Z
batch_001,DOC000003,TEST00000003,Sample Customer 003,ACCOUNT_STATEMENT,2026-09-30,DOC000003.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000003.pdf,eecd9ffaf648015b1dcc0a49067b9724e9a38b609a1174a19046bf6d33a662c7,null,/Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv,2026-10-13T14:41:42.000Z,2026-10-13T14:51:21.115Z
batch_001,DOC000004,TEST00000004,Sample Customer 004,ACCOUNT_STATEMENT,2026-09-30,DOC000004.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000004.pdf,8a1b437face5c2d7feb3fbee111a1b98b3f72afe81110e92fa55279fc7e3ea76,null,/Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv,2026-10-13T14:41:42.000Z,2026-10-13T14:51:21.115Z
batch_001,DOC000005,TEST00000005,Sample Customer 005,ACCOUNT_STATEMENT,2026-09-30,DOC000005.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000005.pdf,e9907b9c7b5e0b6d0836ae782f5a5de81780eb22dbba4802102a97ed69773eb6,null,/Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv,2026-10-13T14:41:42.000Z,2026-10-13T14:51:21.115Z


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT document_id) AS unique_documents,
    COUNT(DISTINCT source_file) AS source_csv_files,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
FROM workspace.migration_dev.bronze_metadata
WHERE batch_id = 'batch_001';

total_rows,unique_documents,source_csv_files,rescued_rows
100,100,1,0
